# Sentinel StatArb — research walkthrough

This notebook reproduces the deterministic research/backtest path used by Sentinel. It compares the baseline signal policy with the volatility-aware dynamic debouncer and inspects the OCO stop-limit configuration.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'python'))

from sentinel_statarb.backtest import run
from sentinel_statarb.strategy import DynamicDebouncer, OCOBracket

## 1. Dynamic confirmation depth

The debouncer increases the number of consecutive confirmations as observed volatility rises.

In [ ]:
debouncer = DynamicDebouncer(base=2, max_confirmations=8, reference_vol=0.002)
[(vol, debouncer.required(vol)) for vol in [0.001, 0.002, 0.005, 0.010]]

## 2. OCO stop-limit levels

With a 5 bps stop and 5 bps limit offset, the stop triggers first; a fill is only allowed when the resulting limit order is marketable.

In [ ]:
bracket = OCOBracket(side=1, entry_spread=100.0, reference_notional=100.0, stop_bps=5.0, limit_bps=5.0)
{'stop': bracket.stop_price, 'limit': bracket.limit_price}

## 3. Deterministic flash-crash comparison

The benchmark uses a fixed seed and synthetic path. It is intended to test execution/risk behavior, not predict live performance.

In [ ]:
baseline = run(seed=7, n=50_000, crash_at=25_000, dynamic=False)
dynamic = run(seed=7, n=50_000, crash_at=25_000, dynamic=True)

for result in (baseline, dynamic):
    print(result)

## 4. Interpretation

Focus on changes in entry count, drawdown, and stop behavior. Treat all values as deterministic benchmark outputs from the repository's synthetic scenario.